# 03 · Model Training (V2)

تدريب الـ Voting Ensemble (5 موديلات) على `final_dataset_v2.csv`.

> **الموديل النهائي محفوظ في `model_defenseV2/models/defence_model_v2.pkl`.**
> هذا النوتبوك للتوثيق وإعادة الإنتاج.

---


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent.parent))

import pandas as pd
import numpy as np
import pickle, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
from scipy.sparse import hstack, csr_matrix

from model_defenseV2.src.features import extract_features, url_text_for_tfidf, FEATURE_NAMES
from model_defenseV2.src import config

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:
df = pd.read_csv('../../data/processed/final_dataset_v2.csv')
texts = df['text'].values
y = 1 - df['Label'].values  # 1=phishing
print(f'Data: {len(df)} | Phishing: {(y==1).sum()} | Safe: {(y==0).sum()}')


## استخراج الخصائص + TF-IDF


In [ ]:
# 39 hand-crafted features
features_list = [extract_features(t) for t in texts]
X_manual = np.array([[f[k] for k in FEATURE_NAMES] for f in features_list])

# Triple TF-IDF
tw = TfidfVectorizer(max_features=2500, ngram_range=(1,3), min_df=2, max_df=0.95, sublinear_tf=True)
X_w = tw.fit_transform(texts)
tc = TfidfVectorizer(analyzer='char_wb', max_features=1500, ngram_range=(2,5), min_df=2, max_df=0.95, sublinear_tf=True)
X_c = tc.fit_transform(texts)
tu = TfidfVectorizer(analyzer='char', max_features=500, ngram_range=(2,4), min_df=2, sublinear_tf=True)
X_u = tu.fit_transform([url_text_for_tfidf(t) for t in texts])

X = hstack([X_w, X_c, X_u, csr_matrix(X_manual)])
print(f'Feature matrix: {X.shape} (word:{X_w.shape[1]} + char:{X_c.shape[1]} + url:{X_u.shape[1]} + manual:{len(FEATURE_NAMES)})')


## تدريب 5 موديلات فردية


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
models = {
    'Logistic Regression': LogisticRegression(C=2.0, max_iter=500, random_state=RANDOM_STATE, class_weight='balanced'),
    'Complement NB': ComplementNB(alpha=0.3),
    'Linear SVM': LinearSVC(C=1.5, max_iter=800, random_state=RANDOM_STATE, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=15, random_state=RANDOM_STATE, class_weight='balanced', n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=4, learning_rate=0.15, random_state=RANDOM_STATE),
}
print(f'{"Model":<22s} {"F1":>7s} {"Recall":>7s} {"Prec":>7s} {"Miss":>5s} {"FP":>4s}')
print('-'*55)
for name, model in models.items():
    preds = cross_val_predict(model, X, y, cv=cv)
    print(f'{name:<22s} {f1_score(y,preds):>7.4f} {recall_score(y,preds):>7.4f} {precision_score(y,preds):>7.4f} {((y==1)&(preds==0)).sum():>5d} {((y==0)&(preds==1)).sum():>4d}')


## Voting Ensemble


In [ ]:
ensemble = VotingClassifier(estimators=[
    ('lr', LogisticRegression(C=2.0, max_iter=500, random_state=RANDOM_STATE, class_weight='balanced')),
    ('cnb', ComplementNB(alpha=0.3)),
    ('svm', LinearSVC(C=1.5, max_iter=800, random_state=RANDOM_STATE, class_weight='balanced')),
    ('rf', RandomForestClassifier(n_estimators=100, max_depth=15, random_state=RANDOM_STATE, class_weight='balanced', n_jobs=-1)),
    ('gb', GradientBoostingClassifier(n_estimators=100, max_depth=4, learning_rate=0.15, random_state=RANDOM_STATE)),
], voting='hard')
yp = cross_val_predict(ensemble, X, y, cv=cv)
print(f'Ensemble | F1:{f1_score(y,yp):.4f} | Recall:{recall_score(y,yp):.4f} | Precision:{precision_score(y,yp):.4f}')
print(f'  Missed: {((y==1)&(yp==0)).sum()} | FP: {((y==0)&(yp==1)).sum()}')


## حفظ الموديل


In [ ]:
ensemble.fit(X, y)
artifact = {
    'model': ensemble,
    'tfidf_word': tw,
    'tfidf_char': tc,
    'tfidf_url': tu,
    'feature_names': FEATURE_NAMES,
    'version': 'v4_v2',
    'training_samples': len(y),
    'config': config.get_config_dict(),
}
with open('../models/defence_model_v2.pkl','wb') as f:
    pickle.dump(artifact, f)
import os
print(f'✅ Saved: defence_model_v2.pkl ({os.path.getsize("../models/defence_model_v2.pkl")/(1024*1024):.1f} MB)')


---
**التالي:** [04 - Evaluation](04_evaluation.ipynb)
